<a href="https://colab.research.google.com/github/cho-hj-dev/YOLOv8-Object-Detection/blob/main/vehicle_damage_detection_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# 실험 A: 5개 클래스 / 200 Epoch / 자동 다운로드
# =========================================================

import os
import shutil
import yaml
import glob
from google.colab import files

print("🚀 1단계: 환경 설정 및 설치 중...")
!pip install ultralytics -q
!pip install roboflow -q

from roboflow import Roboflow
print("🚀 2단계: 데이터 다운로드 중...")
rf = Roboflow(api_key="p2tKf3IyD90vQJxYQGvk")
project = rf.workspace("mystudy-su7rs").project("damage-car-lh7s3-oyhi9")
dataset = project.version(4).download("yolov8")

print("🚀 3단계: 클래스 다이어트 (5개 남기기)...")
dataset_path = dataset.location
# 5개 클래스 모두 포함
keep_classes = ['Missing part', 'Broken part', 'Cracked', 'Dent', 'Corrosion']

def filter_dataset(path, keep_names):
    yaml_path = os.path.join(path, 'data.yaml')
    with open(yaml_path, 'r') as f: data = yaml.safe_load(f)

    old_names = data['names']
    if isinstance(old_names, list): old_names = {i: n for i, n in enumerate(old_names)}
    new_map = {}; new_names_list = []

    for old_idx, name in old_names.items():
        if name in keep_names:
            new_map[old_idx] = len(new_names_list)
            new_names_list.append(name)

    for label_file in glob.glob(os.path.join(path, '**/*.txt'), recursive=True):
        if 'classes.txt' in label_file or 'README' in label_file: continue
        new_lines = []
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts or not parts[0].isdigit(): continue
                if int(parts[0]) in new_map:
                    new_lines.append(f"{new_map[int(parts[0])]} {' '.join(parts[1:])}\n")
        with open(label_file, 'w') as f: f.writelines(new_lines)

    data['nc'] = len(new_names_list); data['names'] = new_names_list
    with open(yaml_path, 'w') as f: yaml.dump(data, f)
    print(f"✨ 완료! 남은 클래스: {new_names_list}")

filter_dataset(dataset_path, keep_classes)

print("🔥 4단계: 학습 시작 (200 Epoch)")
# patience=0으로 설정하여 중간에 멈추지 않고 끝까지 달리기
!yolo task=detect mode=train model=yolov8n.pt data={dataset_path}/data.yaml epochs=200 imgsz=640 plots=True patience=0

print("📦 5단계: 결과물 자동 압축 및 다운로드")
list_of_train_runs = glob.glob('runs/detect/train*')
latest_run = max(list_of_train_runs, key=os.path.getctime)
output_filename = "Result_5Classes_200Epoch"
shutil.make_archive(output_filename, 'zip', latest_run)
files.download(f"{output_filename}.zip")
print("✅ 5개 클래스 학습 완료! 수고하셨습니다.")

🚀 1단계: 환경 설정 및 설치 중...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 143.6 MB/s eta 0:00:00
🚀 2단계: 데이터 다운로드 중...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to damage-car-4 in yolov8:: 100%|██████████| 4564/4564 [00:00<00:00, 6174.26it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 3단계: 클래스 다이어트 (5개 남기기)...
✨ 완료! 남은 클래스: ['Broken part', 'Corrosion', 'Cracked', 'Dent', 'Missing part']
🔥 4단계: 학습 시작 (200 Epoch)
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/damage-car-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 5개 클래스 학습 완료! 수고하셨습니다.
